# exp073_gpu_reproducibility_guard_for_exp063_full_replay train

Train-side LightGBM reproducibility guard using the exp072 full 196-feature exp063 replay cache.

## Contents

1. Setup and configuration
2. Exp072 full replay cache check
3. LightGBM reproducibility guard
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from exp063_full_replay_reproducibility_guard import (
    FULL_REPLAY_TRAIN_FEATURES,
    find_artifact,
    run_reproducibility_guard,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "audit.mode"))
print("Parent:", cfg_get(config, "lineage.parent"))
print("Kernel sources:", cfg_get(config, "runtime.kaggle.kernel_sources"))
print("Active modes:", cfg_get(config, "model.training.active_modes"))


## 2. Exp072 full replay cache check

In [ ]:
cache_path = find_artifact(
    FULL_REPLAY_TRAIN_FEATURES,
    cfg_get(config, "data.exp072_train_feature_cache_local"),
)
print("exp072 exp063 full replay train cache:", cache_path)
preview = pd.read_csv(cache_path, nrows=5, dtype={"id": str, "well": str})
print("Columns:", len(preview.columns))
display(preview)


## 3. LightGBM reproducibility guard

In [ ]:
summary = run_reproducibility_guard(
    output_dir=paths.artifacts_dir,
    cache_path=cfg_get(config, "data.exp072_train_feature_cache_local"),
    modes=cfg_get(config, "model.training.modes", {}),
    active_modes=cfg_get(config, "model.training.active_modes", []),
    n_splits=int(cfg_get(config, "validation.n_folds", 5)),
    fast=bool(cfg_get(config, "audit.fast", False)),
    early_stopping_rounds=int(cfg_get(config, "model.training.early_stopping_rounds", 250)),
    max_rows=cfg_get(config, "model.training.max_rows"),
    max_train_rows=cfg_get(config, "model.training.max_train_rows"),
    save_models=bool(cfg_get(config, "model.training.save_models", True)),
    save_predictions=bool(cfg_get(config, "model.training.save_predictions", True)),
)
print(json.dumps(summary, indent=2))


## 4. Metrics and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "exp063_full_replay_repro_guard_metrics.csv")
by_well = pd.read_csv(paths.artifacts_dir / "exp063_full_replay_repro_guard_by_well.csv")
schema = pd.read_csv(paths.artifacts_dir / "exp063_full_replay_repro_guard_feature_schema.csv")
manifest_path = paths.artifacts_dir / "exp063_full_replay_repro_guard_lgb_models" / "manifest.json"

display(metrics.sort_values(["mode", "model", "fold"]).head(60))
display(metrics[metrics["fold"].astype(str).eq("pooled")].sort_values("rmse_tvt"))
display(by_well.head(30))
print("Feature count:", len(schema))
print("Model manifest:", manifest_path, "exists=", manifest_path.exists())
